# COMPAS Racial Bias: Reproducing the Finding and Mitigating It

## Introduction

COMPAS (Correctional Offender Management Profiling for Alternative Sanctions) is a proprietary risk-assessment algorithm developed by Northpointe (now Equivant) and deployed across the United States criminal justice system. The tool ingests demographic and criminal-history data about a defendant and outputs a recidivism risk score—typically a number from 1 to 10—that categorizes individuals as low, medium, or high risk. These scores were not advisory footnotes. Judges in Florida, Wisconsin, New York, and elsewhere incorporated them directly into sentencing and bail decisions, meaning a number generated by a black-box model could determine whether a human being went home that night or was locked in a cell.

In 2016, ProPublica journalists analyzed more than 7,000 defendants processed through Broward County, Florida's courts and published *Machine Bias*, one of the most consequential pieces of data journalism of the decade. Their central finding: COMPAS was calibrated to be roughly equally accurate across racial groups in the aggregate, yet it failed in systematically asymmetric ways. Black defendants who did *not* reoffend were nearly twice as likely as white defendants to be falsely flagged as high risk (false positive rate: ~45% vs. ~24%). White defendants who *did* reoffend were more likely than Black defendants to be incorrectly labeled low risk (false negative rate: ~48% vs. ~28%). The algorithm's errors were not random noise—they fell along racial lines in ways that systematically disadvantaged Black defendants at the moment of sentencing.

This notebook does three things. First, it reproduces ProPublica's core finding from the public Broward County dataset, validating that the racial disparity in false positive and false negative rates is real and statistically robust—not a reporting artifact. Second, it quantifies that disparity rigorously using confidence intervals and hypothesis tests, so the magnitude of harm is expressed with appropriate uncertainty rather than as a single alarming headline number. Third, it applies a fairness-aware post-processing pipeline (`fairpipe`) to the same data, re-calibrates the score thresholds to equalize error rates across groups, and measures how much of the disparity is recoverable—and at what cost to overall predictive accuracy.

In [ ]:
# Uncomment if running in Colab or Binder
# !pip install fairpipe 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from fairpipe import FairnessAnalyzer, load_data


In [ ]:
# load data directly from the URL 
url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
data = pd.read_csv(url)

# apply ProPublica's documented cleaning criteria 
df = data[
    (data['days_b_screening_arrest'] <= 30) &
    (data['days_b_screening_arrest'] >= -30) &
    (data['is_recid'] != -1) &
    (data['c_charge_degree'] != 'O') &
    (data['score_text'] != 'N/A')
].reset_index(drop=True)

print(f"Dataset: {len(df):,} defendants")
print(f"Recidivism rate: {df['two_year_recid'].mean():.1%}")
print(f"Racial breakdown:\n{df['race'].value_counts()}")

In [ ]:
# Convert COMPAS score to binary prediction (score >= 5 = predicted high risk)
# This mirrors how the score is used in practice
df["y_pred"] = (df["decile_score"] >= 5).astype(int)
df["y_true"] = df["two_year_recid"]

# Focus on the two largest racial groups for statistical power
df_bw = df[df["race"].isin(["African-American", "Caucasian"])].copy()
print(f"Analysis subset: {len(df_bw):,} defendants")
print(f"African-American: {(df_bw['race']=='African-American').sum():,}")
print(f"Caucasian: {(df_bw['race']=='Caucasian').sum():,}")

## Measure the Bias


In [ ]:
THRESHOLD = 0.05

analyzer = FairnessAnalyzer.from_dataframe(
    df_bw,
    y_pred_col="y_pred",
    y_true_col="y_true",
    sensitive_col="race",
    min_group_size=30, # ensure we have enough data for reliable estimates
)

# Demographic parity difference 
dpd = analyzer.demographic_parity_difference(with_ci=True)
passed = dpd.value <= THRESHOLD


print(f"Demographic Parity Difference: {dpd.value:.4f} | Threshold: {THRESHOLD}")
print(f"95% CI: [{dpd.ci[0]:.4f}, {dpd.ci[1]:.4f}]")
print(f"Group prediction rates: {dpd.n_per_group}")
print(f"\n{'✅ PASSED' if passed else '❌ FAILED — pipeline would be blocked'}")


# Equalized odds (captures false positive rate disparity)
eod = analyzer.equalized_odds_difference(with_ci=True)
passed_eod = eod.value <= THRESHOLD
print(f"Equalized Odds Difference: {eod.value:.4f} | Threshold: {THRESHOLD}")
print(f"95% CI: [{eod.ci[0]:.4f}, {eod.ci[1]:.4f}]")
print(f"\n{'✅ PASSED' if passed_eod else '❌ FAILED — pipeline would be blocked'}")


The COMPAS algorithm scores African-American defendants as high-risk at a rate of 24.5% higher than Caucasian defendants. This gap is not explained by actual recidivism rates. The Equalized Odds Difference of 0.2116(95% CI: [0.1870, 0.2525]) tells the more troubling story: among defendants who will not reoffend, an African-American defendant is 21.16 percentage points more likely to be incorrectly labelled as high-risk than a Caucasian defendant in the same situation.

In practice, this means a person who poses no risk loses their liberty at a significantly higher rate depending on their race. Both confidence intervals sit entirely above zero - this is not a statistical artefact. With 3,175 African-American defendants and 2,103 Caucasian defendants in the analysis, the sample size is large enough that these findings are precise to within roughly -/+2.5 percentage points.

## Visualize the Bias

In [ ]:
# False positive and false negative rates by race
groups = df_bw.groupby("race")
rates = {}

for name, group in groups:
    fp = ((group["y_pred"]==1) & (group["y_true"]==0)).sum() / (group["y_true"]== 0).sum()
    fn = ((group["y_pred"]==0) & (group["y_true"]==1)).sum() / (group["y_true"]==1).sum()
    rates[name] = {"False Positive Rate":fp, "False Negative Rate": fn}

rates_df = pd.DataFrame(rates).T
ax = rates_df.plot(kind="bar", figsize=(8,4), color=["#d62728", "#1f77b4"], rot=0)
ax.set_title("COMPAS Error Rates by race\n(False Positive = flagged high risk, did not recidivate; False Negative = flagged low risk, did recidivate)")
ax.set_ylabel("Rate")
ax.set_ylim(0, 0.6)
plt.tight_layout()
plt.savefig("compas_error_rates.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Run the CLI validation
df_bw[["y_pred", "y_true", "race"]].to_csv("compas_bw.csv", index=False)


## Step 2: Apply Mitigation

The question practitioners actually face is: *what do we do about it?*

`fairpipe`'s pipeline module provides a set of bias mitigation transformers that operate
directly on your data before model training. 
Here we apply **Instance Reweighting** —
one of the most well-established pre-processing mitigation techniques. It works by
assigning sample weights that reduce the influence of over-represented group/label
combinations, nudging the model toward more equitable predictions without discarding
any data.

Before applying the transformer, we run fairpipe's **bias detectors** — a suite of
statistical tests that scan the dataset for representation imbalance, proxy variables,
and distributional disparities across sensitive groups. This gives a structured
diagnostic before we intervene.

The configuration is intentionally minimal: one YAML file, one transformer, one
sensitive attribute. In a production setting this config would live in your repository
and be version-controlled alongside your model.

In [ ]:
from fairpipe.pipeline import load_config, build_pipeline, apply_pipeline, run_detectors

# write a minimal config 
config_yaml = """
sensitive: ["race"]
pipeline:
  - name: reweigh
    transformer: "InstanceReweighting"
training:
  method: "reductions"
  target_column: "y_true"
  params:
    constraint: "demographic_parity"
    eps: 0.05
fairness_metric: "equalized_odds_difference"
validation_threshold: 0.05
"""

with open("compas_config.yml", "w") as f:
    f.write(config_yaml)

config = load_config("compas_config.yml")

# Detect bias 
detector_report = run_detectors(df=df_bw, cfg=config)
summary = detector_report.body["summary"]
print("===== Bias Detection Summary ===== ")
print(f"Sensitive attribute: race")
print(f"Disparity flags: {summary['disparity_flags']} (features with significant racial bias)")
print(f"Proxy flags: {summary['proxy_flags']} (features that are strong proxies for race)")
print(f"Representation flags: {summary['representation_flags']} (group size imbalances)")

# Highlight the most import flagged proxies by strength 
proxies = detector_report.body["proxies"]
strong_proxies = sorted(
    [p for p in proxies if p["flagged"]],
    key=lambda x: x["strength"],
    reverse=True
)[:5]

print("\n --- Top 5 Strongest Proxy Features ---")
for p in strong_proxies:
    print(f" {p['feature']:<30} Cramer's V = {p['strength']:.3f}")


# Apply mitigation 
pipeline = build_pipeline(config)
df_mitigated, _ = apply_pipeline(pipeline, df_bw)
print(f"\nMitigation applied. Rows: {len(df_mitigated):,}")


The detector found **28 features with statistically significant racial disparities**
and **23 potential proxy variables** — features that are not race but are strongly
correlated with it and could allow a model to discriminate indirectly.

The top proxies (Cramér's V ≈ 1.0) are identifiers like case numbers, jail dates,
and names — essentially unique keys that encode race perfectly. A model trained on
these features would learn racial patterns even if the `race` column were removed.
This is the proxy discrimination problem in its clearest form.

Instance Reweighting addresses this by rebalancing the influence of each sample
during training — it does not drop these columns, but reduces the weight of
combinations that drive the disparity. In a production pipeline you would also
invoke `ProxyDropper` to remove the highest-strength proxies before training.

## Step 3: Validate the Mitigation

Step 3 compares fairness metrics before and after mitigation using a real trained model — the correct way to evaluate InstanceReweighting, which produces sample weights that must be passed to a classifier at fit time.                                                                                                                                             
We train two logistic regression models on the same five race-excluded features (age, prior counts, juvenile history):                                                                                                                                                                   
     - clf_base : no weights → baseline disparity                                                                                                                                                                       
     - clf_fair : weighted by the InstanceReweighting output → mitigated                                                                                                                                                

Both models generate fresh y_pred columns; FairnessAnalyzer then measures Demographic Parity Difference and Equalized Odds Difference on each, so we can report the actual reduction in bias — and whether it clears the 0.05 threshold.


In [ ]:
from sklearn.linear_model import LogisticRegression

FEATURES = ["age", "priors_count", "juv_fel_count", "juv_misd_count", "juv_other_count"]

# Apply mitigation and unpack correctly
df_mitigated, metadata = apply_pipeline(pipeline, df_bw)
sample_weights = metadata.get("sample_weight", None)

print(f"Sample weights extracted: {sample_weights is not None}")
print(f"Sample weights shape: {sample_weights.shape if sample_weights is not None else 'None'}")
print(f"Sample weights range: [{sample_weights.min():.4f}, {sample_weights.max():.4f}]" 
      if sample_weights is not None else "")

# Train WITHOUT mitigation (baseline)
clf_base = LogisticRegression(max_iter=1000, random_state=42)
clf_base.fit(df_bw[FEATURES], df_bw["y_true"])
df_bw = df_bw.copy()
df_bw["y_pred_model"] = clf_base.predict(df_bw[FEATURES])

# Train WITH mitigation — weights now correctly passed
clf_fair = LogisticRegression(max_iter=1000, random_state=42)
clf_fair.fit(df_mitigated[FEATURES], df_mitigated["y_true"],
             sample_weight=sample_weights)
df_mitigated = df_mitigated.copy()
df_mitigated["y_pred_model"] = clf_fair.predict(df_mitigated[FEATURES])

# Measure fairness on model predictions
analyzer_base = FairnessAnalyzer.from_dataframe(
    df_bw, y_pred_col="y_pred_model", sensitive_col="race", y_true_col="y_true"
)
analyzer_fair = FairnessAnalyzer.from_dataframe(
    df_mitigated, y_pred_col="y_pred_model", sensitive_col="race", y_true_col="y_true"
)

eod_base = analyzer_base.equalized_odds_difference(with_ci=True)
eod_fair = analyzer_fair.equalized_odds_difference(with_ci=True)
dpd_base = analyzer_base.demographic_parity_difference(with_ci=True)
dpd_fair = analyzer_fair.demographic_parity_difference(with_ci=True)

passed_before = eod_base.value <= THRESHOLD
passed_after  = eod_fair.value <= THRESHOLD

print("\n=== Before vs After Mitigation ===")
print(f"{'Metric':<6}  {'Before':>8}  {'After':>8}  {'Change':>8}  {'Threshold':>10}  {'Status'}")
print(f"{'-'*62}")
print(f"{'DPD':<6}  {dpd_base.value:>8.4f}  {dpd_fair.value:>8.4f}  "
      f"{dpd_fair.value - dpd_base.value:>+8.4f}  {'—':>10}")
print(f"{'EOD':<6}  {eod_base.value:>8.4f}  {eod_fair.value:>8.4f}  "
      f"{eod_fair.value - eod_base.value:>+8.4f}  {THRESHOLD:>10.2f}  "
      f"{'✅ PASSED' if passed_after else '❌ FAILED'}")
print()
print(f"Pipeline status BEFORE mitigation: {'✅ PASSED' if passed_before else '❌ FAILED'}")
print(f"Pipeline status AFTER  mitigation: {'✅ PASSED' if passed_after  else '❌ FAILED'}")

In [ ]:
improvement_eod = eod_base.value - eod_fair.value
improvement_pct = (improvement_eod / eod_base.value) * 100

print(f"EOD reduction: {improvement_eod:.4f} ({improvement_pct:.1f}% improvement)")
print(f"Remaining gap to threshold: {eod_fair.value - THRESHOLD:.4f}")
print()
print("Instance Reweighting produced a measurable but partial improvement.")
print("The bias persists because features like priors_count and age are")
print("themselves racially disparate — reweighting alone cannot overcome")
print("structural correlation in the features.")

### What this tells us

Instance Reweighting reduced the Equalized Odds Difference by **0.0102 — a 3.8%
improvement**. The pipeline correctly identifies this as a **❌ FAILED** result,
with a remaining gap of **0.2074** to the 0.05 threshold.

This is the honest outcome. The COMPAS dataset encodes racial disparity not just
in the predictions but in the features themselves - prior counts, age, and juvenile
records all showed statistically significant racial disparities in the detector
output above. A reweighting intervention at training time cannot fully decouple
the model from features that are structurally correlated with race.

Closing the remaining gap would require one or more of:
- **Proxy removal** — dropping or transforming the most racially correlated features
  using `ProxyDropper` before training
- **Stronger constraints** — using `ReductionsWrapper` with an explicit equalized
  odds constraint via Fairlearn
- **Accepting a performance-fairness tradeoff** — a fairer model on this data
  will have lower overall accuracy

This is not a failure of fairpipe. It is fairpipe working exactly as intended —
surfacing a hard problem clearly so practitioners can make an informed decision,
rather than shipping a biased model unknowingly.

## From Notebook to CI/CD Pipeline

Running this analysis manually in a notebook is valuable for understanding.
But in a production ML team, fairness checks need to happen automatically — on every pull request, before every deployment, without relying on anyone remembering to run a notebook.

This is what the `fairpipe` GitHub Action does. Add three lines of YAML to your repository and every PR is automatically checked against your fairness threshold:

```yaml
- uses: SvrusIO/fairpipe-action@v1
  with:
    csv: data/predictions.csv
    y-true: y_true
    y-pred: y_pred
    sensitive: race
    threshold: "0.05"
    fail-on-violation: "true"
```

If the Equalized Odds Difference exceeds 0.05 the PR is blocked. The same metric that flagged a 0.2676 disparity in this notebook would have blocked every COMPAS model deployment automatically - before it reached a judge's courtroom.

The action writes a full fairness report to the GitHub Actions job summary, including metric values, confidence intervals, and group breakdowns, so the evidence is attached to the commit permanently.

→ **[SvrusIO/fairpipe-action](https://github.com/SvrusIO/fairpipe-action)**

## Conclusions

This notebook reproduced and extended ProPublica's 2016 finding using `fairpipe` as the measurement and mitigation framework. The key results:

- The COMPAS algorithm scores Black defendants as high-risk at a rate
  **24.5 percentage points higher** than white defendants (DPD = 0.2451,
  95% CI: [0.2197, 0.2716])
- Among defendants who will not reoffend, Black defendants are **21.2
  percentage points more likely** to be incorrectly labelled high-risk
  (EOD = 0.2116, 95% CI: [0.1870, 0.2525])
- Both confidence intervals sit entirely above zero — this is not noise
- The bias detector identified **28 features with statistically significant
  racial disparities** and **23 proxy variables** — removing the `race`
  column alone would not fix this model
- Instance Reweighting produced a **3.8% reduction in EOD** — a real but
  partial improvement, with a remaining gap of 0.2074 to the 0.05 threshold
- Closing the gap fully requires stronger interventions: proxy removal,
  constraint-based training, or accepting a performance-fairness tradeoff

The deeper point is not about COMPAS specifically. It is that any model making
decisions about people — in hiring, lending, healthcare, or criminal justice —
can carry disparities this large without any single person intending it. The
question is whether your team will find out before or after deployment.

`fairpipe` is built to make sure you find out before.

---

### Try It Yourself

[![Launch in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/SvrusIO/fAIr/main?filepath=case_studies/compas_racial_bias.ipynb)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SvrusIO/fAIr/blob/main/case_studies/compas_racial_bias.ipynb)

```bash
pip install fairpipe
```

**→ [GitHub](https://github.com/SvrusIO/fAIr) · [PyPI](https://pypi.org/project/fairpipe/) · [GitHub Action](https://github.com/SvrusIO/fairpipe-action)**

*Built by [Svrus](https://github.com/SvrusIO) — open-source ML fairness tooling*